In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# dataset -> local disk (fast reads)
LOCAL_DATA = "/tmp/tea_leaf_local"
if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
print("Data ready:", LOCAL_DATA)

# get the module into this session
MODULE_SRC = os.path.join(OUT, "leakage_audit.py")
if os.path.exists(MODULE_SRC):
    shutil.copy(MODULE_SRC, "/content/leakage_audit.py")
    print("Module copied from Drive")
elif os.path.exists("/content/leakage_audit.py"):
    print("Module already in /content")
else:
    raise FileNotFoundError(
        "Put leakage_audit.py in your Drive study folder, "
        "OR drag it into the Colab file panel (left sidebar), then re-run this cell.")

import leakage_audit as la
print("Module imported")

In [ ]:
!pip install -q timm
import timm, torch, torchvision.transforms as T
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
if not os.path.exists(dv1_path):
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters(): p.requires_grad = False

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
print("DINOv1 ready for embedding pass")

In [ ]:
report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 8, dhash_thresh = 8,
    use_embeddings = True,      # set False to skip Cell 2 entirely
    emb_cos_thresh = 0.96,
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
    save_montage_dir = os.path.join(FIGURES_DIR, "dup_clusters"),
)

train_clean, test_clean = la.grouped_split(uf, imgs, seed=42, test_frac=0.2)

In [ ]:
report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 0, dhash_thresh = 0,     # exact canonical only — no similarity chaining
    use_embeddings = True, emb_cos_thresh = 0.97,   # 0.97 is stricter than the 0.96 that gave 169
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
    save_montage_dir = os.path.join(FIGURES_DIR, "dup_clusters_strict"),
)
train_clean, test_clean = la.grouped_split(uf, imgs, seed=42, test_frac=0.2)

In [ ]:
import importlib, leakage_audit
importlib.reload(leakage_audit)          # picks up the new version without a runtime restart
import leakage_audit as la

report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 0, dhash_thresh = 0,        # exact-canonical only (the settings that gave 3.48%)
    use_embeddings = True, emb_cos_thresh = 0.97,
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
)

res = la.inspect_cross_class(
    uf, imgs,
    save_dir = os.path.join(FIGURES_DIR, "cross_class"),
    out_json = os.path.join(RESULTS_DIR, "cross_class_conflicts.json"),
)

In [ ]:
import leakage_audit, importlib, shutil, os
from google.colab import files

print("Python imports leakage_audit from:", leakage_audit.__file__)
print("Now pick the NEW leakage_audit.py you downloaded from the chat:")
up = files.upload()                       # choose the updated file
src = list(up.keys())[0]
if os.path.abspath(src) != os.path.abspath(leakage_audit.__file__):
    shutil.move(src, leakage_audit.__file__)   # force new bytes onto the imported path
importlib.reload(leakage_audit)
import leakage_audit as la
assert hasattr(la, "inspect_cross_class"), "still old — restart runtime and re-run"
print("New version loaded:", leakage_audit.__file__)

In [ ]:
report, uf, imgs = la.run_audit(
    data_root=LOCAL_DATA, seed=42, test_frac=0.2,
    phash_thresh=0, dhash_thresh=0,
    use_embeddings=True, emb_cos_thresh=0.97,
    out_json=os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
assert getattr(uf, "provenance", None) is not None, "old run_audit ran — G1 didn't take"

res = la.inspect_cross_class(
    uf, imgs,
    save_dir=os.path.join(FIGURES_DIR, "cross_class"),
    out_json=os.path.join(RESULTS_DIR, "cross_class_conflicts.json"),
)

In [ ]:
# Decide: real cross-class duplicates vs perceptual-hash collisions
import numpy as np, os, json
from PIL import Image

def _gray(im, sz=64):
    return np.asarray(im.convert('L').resize((sz, sz)), float)

def best_ncc(pa, pb, sz=64):
    """Brightness/orientation-invariant similarity. ~1 = same image, ~0 = different."""
    a = _gray(Image.open(pa), sz); av = a - a.mean(); na = np.linalg.norm(av) + 1e-9
    b_im = Image.open(pb); best = -1.0
    for base in (b_im, b_im.transpose(Image.FLIP_LEFT_RIGHT)):
        for rot in (0, 90, 180, 270):
            b = _gray(base.rotate(rot), sz); bv = b - b.mean()
            best = max(best, float((av*bv).sum() / (na*(np.linalg.norm(bv)+1e-9))))
    return best

REAL, COLL, MIXED = [], [], []
real_conflict_paths = set()
healthy_report = []

for e in res["label_conflicts"]:
    paths = e.get("conflict_paths", e["paths"])
    # all cross-class pairs
    pair_ncc = []
    for i in range(len(paths)):
        for j in range(i+1, len(paths)):
            ci = os.path.basename(os.path.dirname(paths[i]))
            cj = os.path.basename(os.path.dirname(paths[j]))
            if ci != cj:
                try: s = best_ncc(paths[i], paths[j])
                except Exception: s = float('nan')
                pair_ncc.append((s, paths[i], paths[j], ci, cj))
    if not pair_ncc: continue
    sims = [s for s,_,_,_,_ in pair_ncc if not np.isnan(s)]
    mx, mn = max(sims), min(sims)
    # confirmed real cross-class dup pairs
    real_pairs = [(s,pi,pj,ci,cj) for s,pi,pj,ci,cj in pair_ncc if s >= 0.90]
    for _,pi,pj,_,_ in real_pairs:
        real_conflict_paths.add(pi); real_conflict_paths.add(pj)
    if mn >= 0.90:   REAL.append((e, mx, mn))
    elif mx < 0.70:  COLL.append((e, mx, mn))
    else:            MIXED.append((e, mx, mn))
    if any('healthy' in c.lower() for c in e["classes"]):
        healthy_report.append((e["classes"], round(mx,3), round(mn,3), real_pairs))

print("="*60)
print("PIXEL-LEVEL VERDICT on the 66 cross-class clusters")
print("="*60)
print(f"REAL duplication (min pairwise NCC >= 0.90) : {len(REAL)}")
print(f"COLLISION (max pairwise NCC < 0.70)         : {len(COLL)}")
print(f"MIXED (some real, some collided)            : {len(MIXED)}")
print(f"Images in confirmed cross-class dup pairs    : {len(real_conflict_paths)}")
print("-"*60)
print("HEALTHY-involving clusters (decisive for binary labels):")
for classes, mx, mn, rp in healthy_report:
    tag = "REAL DUP" if mx >= 0.90 else "collision"
    print(f"  {' vs '.join(classes)}  max={mx} min={mn}  -> {tag}")
    for s,pi,pj,ci,cj in rp:
        print(f"     {ci}  ==  {cj}   (NCC={s:.3f})")
        print(f"        {os.path.relpath(pi)}")
        print(f"        {os.path.relpath(pj)}")

with open(os.path.join(RESULTS_DIR, "real_cross_class_conflicts.json"), "w") as f:
    json.dump({"n_real": len(REAL), "n_collision": len(COLL), "n_mixed": len(MIXED),
               "real_conflict_paths": sorted(real_conflict_paths)}, f, indent=2)
print(f"\nSaved -> real_cross_class_conflicts.json")

In [ ]:
# ============================================================
# Cell A — Recovery / setup (run first in any new session)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ---- paths ----
TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- copy dataset to local disk (fast reads) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
    print(" Copied")
else:
    print(" Local cache exists")

# ---- classes ----
classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES   = len(classes)
CLASS_TO_IDX  = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

# ---- load DINOv1 ViT-S/16 ----
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

# ---- attention hook on last block (stores CLS->patch attention) ----
_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = (cls_q @ k.transpose(-2, -1)) * scale
    cls_attn = cls_attn.softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()   # [1, heads, patches]
    attn_full = (q @ k.transpose(-2, -1)) * scale
    attn_full = attn_full.softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

# ---- transforms ----
tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

# ---- attention geometry ----
DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))   # top-10% patches

# ---- sanity check the hook ----
with torch.no_grad():
    _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
_cls = _dv1_store['cls_attn']
print(f"Attention hook OK: {_cls.shape} (heads={_cls.shape[1]}, patches={_cls.shape[2]})")

# ---- collect all images (LOCAL paths) ----
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# ---- datasets ----
class SimpleDS(Dataset):
    """Training set: images + provided (pseudo) labels."""
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    """Test set: images + true is_diseased labels."""
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.imgs[idx]['is_diseased']

print(f"\n Ready: {len(all_images)} images, {NUM_CLASSES} classes")
n_dis = sum(x['is_diseased'] for x in all_images)
print(f" Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")
print(f"\nExisting results on Drive:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith('.json'):
        print(f"   {f}")

In [ ]:
# ============================================================
# Cell B' — Grouped-split confirmation of the pipeline (3 seeds)
# Leakage-safe: duplicate clusters never straddle train/test.
# ============================================================
import numpy as np, random, json, os
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from PIL import Image
from tqdm import tqdm
import leakage_audit as la

# sanity: audit outputs must be in memory
assert 'uf' in dir() and 'imgs' in dir(), "Run the audit cell (G2) first to get uf, imgs"
assert getattr(uf, "provenance", None) is not None, "uf has no provenance — re-run G2"

DROP = False   # 0 real cross-class conflicts confirmed -> nothing to drop
drop_ids = la.flagged_to_drop(uf, imgs, drop_conflicts=True,
                              dedup_same_class=True) if DROP else set()

print("="*60)
print("CONFIRMATION: grouped-split pipeline (3 seeds)")
print("="*60)

N_SEEDS = 3
grouped_results = []

for seed in range(N_SEEDS):
    print(f"\n--- Seed {seed+1}/{N_SEEDS} ---")
    random.seed(seed*1000 + 42)

    # leakage-safe split (whole duplicate clusters kept on one side)
    train_s, test_s = la.grouped_split(uf, imgs, seed=seed*1000+42, test_frac=0.2)
    if DROP:
        train_s = [x for x in train_s if id(x) not in drop_ids]
        test_s  = [x for x in test_s  if id(x) not in drop_ids]

    # --- UAHS head discovery ---
    pool = random.sample(train_s, min(200, len(train_s)))
    head_ent = {h: [] for h in range(N_HEADS)}
    with torch.no_grad():
        for s in pool:
            try:
                im = Image.open(s['path']).convert('RGB')
                _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
                attn = _dv1_store['cls_attn'][0].numpy()
                for h in range(N_HEADS):
                    a = attn[h]; a = a/(a.sum()+1e-10)
                    head_ent[h].append(-(a*np.log(a+1e-10)).sum())
            except Exception:
                continue
    sel_heads = sorted(range(N_HEADS), key=lambda h: -np.std(head_ent[h]))[:2]
    print(f"  UAHS heads: {sel_heads}")

    # --- score all training images (attention concentration) ---
    scores = []
    with torch.no_grad():
        for s in tqdm(train_s, desc=f"  Scoring seed {seed+1}"):
            try:
                im = Image.open(s['path']).convert('RGB')
                _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
                attn = _dv1_store['cls_attn'][0].numpy()
                a = attn[sel_heads].mean(axis=0); a = a/(a.sum()+1e-10)
                scores.append(np.sort(a)[::-1][:topk].sum())
            except Exception:
                scores.append(0.0)
    scores = np.array(scores)
    true_bin = np.array([s['is_diseased'] for s in train_s])

    # --- pseudo-labels via percentile threshold ---
    h_frac = (true_bin == 0).mean()
    thresh = np.percentile(scores, h_frac*100)
    pseudo = (scores > thresh).astype(int)
    p_acc = accuracy_score(true_bin, pseudo)
    p_f1  = f1_score(true_bin, pseudo)
    print(f"  Pseudo acc: {p_acc:.4f}, F1: {p_f1:.4f}")

    # --- train MobileNetV3 on pseudo-labels ---
    ds_t = SimpleDS(train_s, pseudo.tolist(), tf_train)
    ds_e = TestDS(test_s, tf_test)
    loader_t = DataLoader(ds_t, batch_size=64, shuffle=True,  num_workers=2)
    loader_e = DataLoader(ds_e, batch_size=64, shuffle=False, num_workers=2)
    test_true = np.array([s['is_diseased'] for s in test_s])

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
    crit  = nn.CrossEntropyLoss()

    model.train()
    for ep in range(30):
        for xb, yb in loader_t:
            xb, yb = xb.to(device), yb.to(device).long()
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (ep+1) % 10 == 0:
            print(f"    Epoch {ep+1}/30")

    # --- evaluate ---
    model.eval(); probs, preds = [], []
    with torch.no_grad():
        for xb, _ in loader_e:
            out = model(xb.to(device))
            probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
            preds.append(out.argmax(1).cpu().numpy())
    probs = np.concatenate(probs); preds = np.concatenate(preds)

    auc = roc_auc_score(test_true, probs)
    acc = accuracy_score(test_true, preds)
    f1  = f1_score(test_true, preds)
    print(f"  ★ AUC={auc:.4f}, Acc={acc:.4f}, F1={f1:.4f}  "
          f"(train={len(train_s)}, test={len(test_s)})")

    grouped_results.append({'seed': seed, 'uahs_heads': sel_heads,
                            'pseudo_acc': float(p_acc), 'pipeline_auc': float(auc),
                            'pipeline_acc': float(acc), 'pipeline_f1': float(f1)})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# --- summary ---
aucs = [r['pipeline_auc'] for r in grouped_results]
accs = [r['pipeline_acc'] for r in grouped_results]
f1s  = [r['pipeline_f1']  for r in grouped_results]
paccs= [r['pseudo_acc']   for r in grouped_results]

print(f"\n{'='*60}\nGROUPED-SPLIT SUMMARY (mean ± std)\n{'='*60}")
print(f"  Pipeline AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Pipeline Acc:     {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"  Pipeline F1:      {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  Pseudo-label Acc: {np.mean(paccs):.4f} ± {np.std(paccs):.4f}")

with open(os.path.join(RESULTS_DIR, 'multi_run_pipeline_grouped.json'), 'w') as f:
    json.dump({'runs': grouped_results,
               'summary': {'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
                           'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
                           'f1_mean':  float(np.mean(f1s)),  'f1_std':  float(np.std(f1s)),
                           'pseudo_acc_mean': float(np.mean(paccs)),
                           'pseudo_acc_std':  float(np.std(paccs))},
               'dropped_flagged': bool(DROP)}, f, indent=2)
print("Saved -> multi_run_pipeline_grouped.json")

In [ ]:
# ============================================================
# Cell A — Setup (run first in any new session)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ---- paths ----
TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (fast reads) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
if not os.path.isdir(LOCAL_DATA):
    print("Copying dataset to local disk (~2 min)...")
    shutil.copytree(TEA_LEAF_ROOT, LOCAL_DATA)
    print(" Copied")
else:
    print(" Local cache exists")

# ---- classes ----
classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

# ---- DINOv1 ViT-S/16 ----
dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

# ---- attention hook (CLS -> patch attention from last block) ----
_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

# ---- transforms ----
tf_dv1 = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_train = T.Compose([
    T.Resize((224, 224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(20), T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
tf_test = T.Compose([
    T.Resize((224, 224)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

DV1_GRID = 14
N_HEADS  = 6
topk     = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1, 3, 224, 224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

# ---- image list ----
all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname),
                'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1,
            })

# ---- datasets ----
class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs = imgs; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.labels[idx]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs; self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        img = Image.open(self.imgs[idx]['path']).convert('RGB')
        return self.transform(img), self.imgs[idx]['is_diseased']

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
# ============================================================
# Cell G1 — upload leakage_audit.py (the updated version)
# ============================================================
import os, shutil
from google.colab import files

TARGET = "/content/leakage_audit.py"
if not os.path.exists(TARGET):
    print("Select the updated leakage_audit.py you downloaded from chat:")
    up = files.upload()
    src = list(up.keys())[0]
    if os.path.abspath(src) != os.path.abspath(TARGET):
        shutil.move(src, TARGET)

import importlib, sys
sys.modules.pop("leakage_audit", None)      # drop any stale cached module
import leakage_audit as la
importlib.reload(la)

assert hasattr(la, "inspect_cross_class"), \
    "Old version on disk. Delete /content/leakage_audit.py and re-run this cell."
print("Module OK:", la.__file__)

In [ ]:
# ============================================================
# Cell G2 — leakage audit (produces uf + imgs for the grouped split)
# ============================================================
report, uf, imgs = la.run_audit(
    data_root      = LOCAL_DATA,
    seed           = 42, test_frac = 0.2,
    phash_thresh   = 0, dhash_thresh = 0,        # exact-canonical only (the settings that gave 3.48%)
    use_embeddings = True, emb_cos_thresh = 0.97,
    out_json       = os.path.join(RESULTS_DIR, "leakage_audit.json"),
)
assert getattr(uf, "provenance", None) is not None, "old run_audit ran — fix G1"
print("\nuf + imgs ready for grouped_split")

In [ ]:
# ============================================================
# Cell B' — Grouped-split pipeline (3 seeds). Resumable per seed.
# ============================================================
import numpy as np, random, json, os
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from PIL import Image
from tqdm import tqdm

assert 'uf' in dir() and 'imgs' in dir(), "Run G2 first"
assert getattr(uf, "provenance", None) is not None, "uf has no provenance — re-run G2"

DROP = False   # 0 real cross-class conflicts confirmed -> nothing to drop
drop_ids = la.flagged_to_drop(uf, imgs, drop_conflicts=True,
                              dedup_same_class=True) if DROP else set()

CKPT = os.path.join(RESULTS_DIR, 'multi_run_grouped_partial.json')
grouped_results = []
if os.path.exists(CKPT):
    with open(CKPT) as f:
        grouped_results = json.load(f)
    print(f"Resuming: {len(grouped_results)} seed(s) already done")

print("="*60)
print("CONFIRMATION: grouped-split pipeline (3 seeds)")
print("="*60)

N_SEEDS = 3
done_seeds = {r['seed'] for r in grouped_results}

for seed in range(N_SEEDS):
    if seed in done_seeds:
        print(f"\n--- Seed {seed+1}/{N_SEEDS}: already done, skipping ---")
        continue
    print(f"\n--- Seed {seed+1}/{N_SEEDS} ---")
    random.seed(seed*1000 + 42)

    train_s, test_s = la.grouped_split(uf, imgs, seed=seed*1000+42, test_frac=0.2)
    if DROP:
        train_s = [x for x in train_s if id(x) not in drop_ids]
        test_s  = [x for x in test_s  if id(x) not in drop_ids]

    # UAHS head discovery
    pool = random.sample(train_s, min(200, len(train_s)))
    head_ent = {h: [] for h in range(N_HEADS)}
    with torch.no_grad():
        for s in pool:
            try:
                im = Image.open(s['path']).convert('RGB')
                _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
                attn = _dv1_store['cls_attn'][0].numpy()
                for h in range(N_HEADS):
                    a = attn[h]; a = a/(a.sum()+1e-10)
                    head_ent[h].append(-(a*np.log(a+1e-10)).sum())
            except Exception:
                continue
    sel_heads = sorted(range(N_HEADS), key=lambda h: -np.std(head_ent[h]))[:2]
    print(f"  UAHS heads: {sel_heads}")

    # score training images
    scores = []
    with torch.no_grad():
        for s in tqdm(train_s, desc=f"  Scoring seed {seed+1}"):
            try:
                im = Image.open(s['path']).convert('RGB')
                _ = dino_v1(tf_dv1(im).unsqueeze(0).to(device))
                attn = _dv1_store['cls_attn'][0].numpy()
                a = attn[sel_heads].mean(axis=0); a = a/(a.sum()+1e-10)
                scores.append(np.sort(a)[::-1][:topk].sum())
            except Exception:
                scores.append(0.0)
    scores   = np.array(scores)
    true_bin = np.array([s['is_diseased'] for s in train_s])

    # pseudo-labels
    h_frac = (true_bin == 0).mean()
    thresh = np.percentile(scores, h_frac*100)
    pseudo = (scores > thresh).astype(int)
    p_acc  = accuracy_score(true_bin, pseudo)
    p_f1   = f1_score(true_bin, pseudo)
    print(f"  Pseudo acc: {p_acc:.4f}, F1: {p_f1:.4f}")

    # train MobileNetV3
    loader_t = DataLoader(SimpleDS(train_s, pseudo.tolist(), tf_train),
                          batch_size=64, shuffle=True,  num_workers=2)
    loader_e = DataLoader(TestDS(test_s, tf_test),
                          batch_size=64, shuffle=False, num_workers=2)
    test_true = np.array([s['is_diseased'] for s in test_s])

    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=1e-3)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
    crit  = nn.CrossEntropyLoss()

    model.train()
    for ep in range(30):
        for xb, yb in loader_t:
            xb, yb = xb.to(device), yb.to(device).long()
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (ep+1) % 10 == 0:
            print(f"    Epoch {ep+1}/30")

    model.eval(); probs, preds = [], []
    with torch.no_grad():
        for xb, _ in loader_e:
            out = model(xb.to(device))
            probs.append(torch.softmax(out, 1)[:, 1].cpu().numpy())
            preds.append(out.argmax(1).cpu().numpy())
    probs = np.concatenate(probs); preds = np.concatenate(preds)

    auc = roc_auc_score(test_true, probs)
    acc = accuracy_score(test_true, preds)
    f1  = f1_score(test_true, preds)
    print(f"  * AUC={auc:.4f}, Acc={acc:.4f}, F1={f1:.4f}  "
          f"(train={len(train_s)}, test={len(test_s)})")

    grouped_results.append({'seed': seed, 'uahs_heads': sel_heads,
                            'pseudo_acc': float(p_acc), 'pipeline_auc': float(auc),
                            'pipeline_acc': float(acc), 'pipeline_f1': float(f1)})
    with open(CKPT, 'w') as f:            # checkpoint after every seed
        json.dump(grouped_results, f, indent=2)
    print("  (checkpointed)")

    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ---- summary ----
aucs  = [r['pipeline_auc'] for r in grouped_results]
accs  = [r['pipeline_acc'] for r in grouped_results]
f1s   = [r['pipeline_f1']  for r in grouped_results]
paccs = [r['pseudo_acc']   for r in grouped_results]

print(f"\n{'='*60}\nGROUPED-SPLIT SUMMARY (mean ± std)\n{'='*60}")
print(f"  Pipeline AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  Pipeline Acc:     {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"  Pipeline F1:      {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  Pseudo-label Acc: {np.mean(paccs):.4f} ± {np.std(paccs):.4f}")

with open(os.path.join(RESULTS_DIR, 'multi_run_pipeline_grouped.json'), 'w') as f:
    json.dump({'runs': grouped_results,
               'summary': {'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
                           'acc_mean': float(np.mean(accs)), 'acc_std': float(np.std(accs)),
                           'f1_mean':  float(np.mean(f1s)),  'f1_std':  float(np.std(f1s)),
                           'pseudo_acc_mean': float(np.mean(paccs)),
                           'pseudo_acc_std':  float(np.std(paccs))},
               'dropped_flagged': bool(DROP)}, f, indent=2)
print("Saved -> multi_run_pipeline_grouped.json")